## 1. Setup

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.paths import PROCESSED_PATH
from src.optimization import (
    annualized_arithmetic_mu,
    build_constraints,
    compute_turnover,
    portfolio_return,
    portfolio_sharpe,
    portfolio_volatility,
    shrink_mu,
    solve_max_return,
    solve_max_sharpe,
    solve_min_variance,
    solve_min_variance_for_target_return,
    solve_risk_parity,
)

## 2. Paths & Config

In [ ]:
PROJECT_ROOT = Path("..")
DATA_PATH = PROCESSED_PATH

from src.risk import RISK_FREE_RATE  # ECB deposit rate, single source in src/risk.py
TRADING_DAYS = 252

N_SENSITIVITY_RUNS = 30
SENSITIVITY_SHOCK_STD = 0.01

MAX_MISSING_PCT = 0.05
MU_SHRINKAGE = 0.4
CLASS_CAP_MULTIPLIER = 1.5
INDIVIDUAL_CAP_MULTIPLIER = 2.5
INDIVIDUAL_CAP_FLOOR = 0.03
N_FRONTIER_POINTS = 40

## 3. Optimization Engine

All solvers live in `src/optimization.py` (unit-tested in `tests/test_optimization.py`): long-only SLSQP with a fully-invested constraint plus per-position bounds and asset-class caps.

## 4. Load Returns & Estimate mu / Sigma

In [ ]:
asset_returns = pd.read_csv(DATA_PATH / "asset_returns_eur.csv", index_col=0, parse_dates=True)
asset_returns = asset_returns.dropna(axis=1, how="all")

tickers = asset_returns.columns

# Arithmetic, not geometric (CAGR): mean-variance needs E[w'r] = w'mu, which only
# holds for arithmetic expectations. The gap matters for volatile assets
# (roughly sigma^2 / 2 per year). Realized performance (Notebooks 03/07) stays geometric.
mu = annualized_arithmetic_mu(asset_returns, TRADING_DAYS)
mu = mu.reindex(tickers)
mu_shrunk = shrink_mu(mu, MU_SHRINKAGE)

cov = asset_returns.cov() * TRADING_DAYS
cov = cov.reindex(index=tickers, columns=tickers)

mu.sort_values(ascending=False)

## 5. Data Quality Check

In [ ]:
# Notebook 03 already fills gaps with 0.0, so NaNs can't show up here. The
# real signal is the share of zero-return days (holidays / exchange closures,
# forward-filled prices), which slightly dampens volatilities and covariances.
missing_pct = asset_returns.isna().mean()
assert missing_pct.max() == 0, f"Unexpected NaNs in asset returns: {missing_pct[missing_pct > 0].to_dict()}"

zero_share = (asset_returns == 0).mean().sort_values(ascending=False)
print("Share of zero-return days (top 5):")
print(zero_share.head())

thin_assets = zero_share[zero_share > MAX_MISSING_PCT].index.tolist()
if thin_assets:
    print(f"Warning: {thin_assets} have >{MAX_MISSING_PCT:.0%} zero-return days "
          "(stale or missing prices) — their volatility is understated.")

cov_eigenvalues = np.linalg.eigvalsh(cov.values)
if cov_eigenvalues.min() < -1e-8:
    raise ValueError(
        f"Covariance matrix is not positive semi-definite (min eigenvalue "
        f"{cov_eigenvalues.min():.2e}) — likely caused by inconsistent missing-"
        "data patterns across assets. Align the return history before optimizing."
    )

print("Covariance matrix OK (positive semi-definite).")

## 6. Load Target Weights & Asset Classes

In [ ]:
holdings = pd.read_csv(DATA_PATH / "current_holdings_trackable.csv")
target_weights_raw = pd.read_csv(DATA_PATH / "actual_vs_target_weights.csv")

asset_class_map = (
    holdings[["yf_ticker", "asset_class"]]
    .dropna(subset=["yf_ticker"])
    .drop_duplicates(subset="yf_ticker")
    .set_index("yf_ticker")["asset_class"]
    .reindex(tickers)
)

In [ ]:
current_target = (
    target_weights_raw
    .merge(holdings[["symbol", "yf_ticker"]], on="symbol", how="left")
    .dropna(subset=["yf_ticker"])
    .set_index("yf_ticker")[["actual_weight", "target_weight"]]
    .astype(float)
    .reindex(tickers)
    .fillna(0.0)
)
current_target

## 7. Weight Bounds

In [ ]:
individual_caps = (
    (current_target["target_weight"] * INDIVIDUAL_CAP_MULTIPLIER)
    .clip(lower=INDIVIDUAL_CAP_FLOOR, upper=1.0)
)
bounds = [(0.0, individual_caps[t]) for t in tickers]

class_target_weight = current_target["target_weight"].groupby(asset_class_map).sum()
class_aggregate_caps = (class_target_weight * CLASS_CAP_MULTIPLIER).clip(upper=1.0).to_dict()

class_masks = {
    asset_class: (asset_class_map.values == asset_class)
    for asset_class in asset_class_map.unique()
}

constraints = build_constraints(class_masks, class_aggregate_caps)

## 8. Equal-Weight Portfolio

In [ ]:
n_assets = len(tickers)
weights_equal = pd.Series(np.repeat(1 / n_assets, n_assets), index=tickers)
weights_equal

## 9. Minimum-Variance Portfolio

In [ ]:
weights_min_var = pd.Series(
    solve_min_variance(cov.values, bounds, constraints),
    index=tickers
)
weights_min_var[weights_min_var > 1e-4].sort_values(ascending=False)

## 10. Maximum-Sharpe Portfolio

In [ ]:
weights_max_sharpe = pd.Series(
    solve_max_sharpe(mu.values, cov.values, bounds, constraints, RISK_FREE_RATE),
    index=tickers
)
weights_max_sharpe[weights_max_sharpe > 1e-4].sort_values(ascending=False)

## 11. Maximum-Sharpe Portfolio with shrinkage

In [ ]:
weights_max_sharpe_shrunk = pd.Series(
    solve_max_sharpe(mu_shrunk.values, cov.values, bounds, constraints, RISK_FREE_RATE),
    index=tickers
)
weights_max_sharpe_shrunk[weights_max_sharpe_shrunk > 1e-4].sort_values(ascending=False)

## 12. Risk-Parity Portfolio

Equal risk contributions are usually **not** reachable here: small positions hit their caps (`INDIVIDUAL_CAP_FLOOR` and the per-asset caps above), so the solver returns the portfolio whose risk contributions are *as equal as the caps allow* — risk-parity-like, not true risk parity.

In [ ]:
weights_risk_parity = pd.Series(
    solve_risk_parity(cov.values, bounds, constraints),
    index=tickers
)
weights_risk_parity[weights_risk_parity > 1e-4].sort_values(ascending=False)

## 13. Compare Against Current & Target Weights

In [ ]:
weights_summary = pd.DataFrame({
    "Current (Actual)": current_target["actual_weight"],
    "Strategic Target": current_target["target_weight"],
    "Equal Weight": weights_equal,
    "Min-Variance": weights_min_var,
    "Max-Sharpe": weights_max_sharpe,
    "Max-Sharpe (Shrunk mu)": weights_max_sharpe_shrunk,
    "Risk-Parity": weights_risk_parity,
})

def summarize_portfolio(weights, mu, cov, risk_free_rate=RISK_FREE_RATE):
    w = weights.reindex(tickers).fillna(0.0).values
    return {
        "Expected Return": portfolio_return(w, mu.values),
        "Expected Volatility": portfolio_volatility(w, cov.values),
        "Expected Sharpe": portfolio_sharpe(w, mu.values, cov.values, risk_free_rate),
    }

ex_ante_metrics = pd.DataFrame({
    strategy: summarize_portfolio(weights_summary[strategy], mu, cov)
    for strategy in weights_summary.columns
}).T

turnover_to_reach = pd.Series({
    strategy: compute_turnover(weights_summary["Current (Actual)"], weights_summary[strategy])
    for strategy in weights_summary.columns
    if strategy != "Current (Actual)"
}, name="Turnover to Reach (from Current)")

ex_ante_metrics = ex_ante_metrics.join(turnover_to_reach)
ex_ante_metrics

## 14. Efficient Frontier

In [ ]:
# Full minimum-variance frontier under the caps: from the lowest to the highest
# expected return reachable with the same bounds/constraints. A grid spanning
# mu.min()..mu.max() (single worst/best asset) is mostly infeasible under the
# caps, and those points would silently drop out. Points below the
# min-variance return form the inefficient lower branch (flagged, drawn dashed).
min_var_return = portfolio_return(weights_min_var.values, mu.values)
min_return_weights = solve_max_return(-mu.values, bounds, constraints)
min_return = portfolio_return(min_return_weights, mu.values)
max_return = portfolio_return(solve_max_return(mu.values, bounds, constraints), mu.values)

target_returns = np.linspace(min_return, max_return, N_FRONTIER_POINTS)
frontier_points = []
failed_targets = []
x0 = min_return_weights  # warm start: each point starts from the previous one

for target_return in target_returns:
    weights = solve_min_variance_for_target_return(
        mu.values, cov.values, bounds, constraints, target_return, x0=x0
    )
    if weights is None:
        failed_targets.append(target_return)
        continue
    x0 = weights
    frontier_points.append({
        "Expected Return": target_return,
        "Expected Volatility": portfolio_volatility(weights, cov.values),
        "Efficient": target_return >= min_var_return - 1e-9
    })

print(f"Frontier: {len(frontier_points)}/{N_FRONTIER_POINTS} points solved "
      f"({min_return:.1%} to {max_return:.1%}, efficient from {min_var_return:.1%})")
assert len(failed_targets) <= N_FRONTIER_POINTS // 10, f"{len(failed_targets)} frontier points failed"

frontier_df = pd.DataFrame(frontier_points)
frontier_df

## 15. Sensitivity to Expected Return Assumptions

In [ ]:
# How much does the Max-Sharpe allocation move if mu is only slightly wrong?
# mu is the least reliable input here (covariance is comparatively stable) —
# this is the standard critique of mean-variance optimization.
rng = np.random.default_rng(seed=42)
sensitivity_runs = []

for run in range(N_SENSITIVITY_RUNS):
    mu_shocked = mu.values + rng.normal(0, SENSITIVITY_SHOCK_STD, size=len(mu))
    weights_shocked = solve_max_sharpe(mu_shocked, cov.values, bounds, constraints, RISK_FREE_RATE)
    sensitivity_runs.append(weights_shocked)

sensitivity_df = pd.DataFrame(sensitivity_runs, columns=tickers)

sensitivity_summary = pd.DataFrame({
    "Base Weight (Max-Sharpe)": weights_max_sharpe,
    "Mean Across Shocks": sensitivity_df.mean(),
    "Std Across Shocks": sensitivity_df.std(),
    "Min Across Shocks": sensitivity_df.min(),
    "Max Across Shocks": sensitivity_df.max(),
})

sensitivity_summary.sort_values("Std Across Shocks", ascending=False)

## 16. Visualization

The frontier is computed **with** the same individual and asset-class caps as the optimized portfolios. Equal Weight ignores those caps (1/N in every position, single stocks included), so it can sit above the constrained frontier — it is not a contradiction.

In [ ]:
plt.figure(figsize=(9, 6))
# Drawn up to the Max-Sharpe portfolio: beyond it the caps bind and the
# frontier only adds volatility for almost no extra return (flat tail).
max_sharpe_return = ex_ante_metrics.loc["Max-Sharpe", "Expected Return"]
efficient = frontier_df[frontier_df["Efficient"] & (frontier_df["Expected Return"] <= max_sharpe_return + 1e-9)]
inefficient = frontier_df[~frontier_df["Efficient"]]
plt.plot(efficient["Expected Volatility"], efficient["Expected Return"],
         color="black", label="Efficient Frontier")
plt.plot(inefficient["Expected Volatility"], inefficient["Expected Return"],
         color="black", linestyle="--", alpha=0.5, label="Inefficient branch")

for strategy in weights_summary.columns:
    row = ex_ante_metrics.loc[strategy]
    plt.scatter(row["Expected Volatility"], row["Expected Return"], s=60, label=strategy)

plt.xlabel("Expected Volatility (annualized)")
plt.ylabel("Expected Return (annualized)")
plt.title("Efficient Frontier vs. Current / Target / Optimized Portfolios")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plot_df = weights_summary[weights_summary.abs().sum(axis=1) > 1e-4]

plot_df.plot(kind="bar", figsize=(14, 6))
plt.title("Portfolio Weights by Strategy")
plt.ylabel("Weight")
plt.xlabel("Asset")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 17. Export

In [ ]:
weights_summary.to_csv(DATA_PATH / "optimization_weights_summary.csv")
ex_ante_metrics.to_csv(DATA_PATH / "optimization_expected_metrics.csv")
sensitivity_summary.to_csv(DATA_PATH / "optimization_sensitivity.csv")
frontier_df.to_csv(DATA_PATH / "optimization_efficient_frontier.csv")

print("All optimization exports saved.")